In [ ]:
import pandas as pd
import json

def enrich_feature_meta(csv_path: str, json_path: str, output_path: str):
    # 读取 CSV 数据
    df = pd.read_csv(csv_path)
    
    # 替换 DataFrame 列名中的 '.' 为 '_'
    df.columns = [col.replace('.', '_') for col in df.columns]

    # 读取 JSON 文件并将 key 中的 '-' 替换为 '_'
    with open(json_path, 'r') as f:
        raw_meta = json.load(f)

    # 替换 JSON 的 key
    original_meta = {k.replace('-', '_'): v for k, v in raw_meta.items()}

    # 构建 enriched metadata
    enriched_meta = {}
    for col in df.columns:
        desc = original_meta.get(col, "")  # 若 JSON 中无该列，desc设为空字符串
        col_data = df[col]

        meta = {
            "description": desc
        }

        # 判断类型
        if col_data.nunique() == len(col_data):  # 每个值都不同
            meta["type"] = "unique_identifier"
            meta["unique_values"] = int(col_data.nunique())
        elif col_data.dtype == object or col_data.dtype.name == "category":
            meta["type"] = "categorical"
            meta["values"] = sorted(col_data.dropna().unique().tolist())
        else:
            meta["type"] = "numerical"

        enriched_meta[col] = meta

    # 保存为新 JSON 文件
    with open(output_path, 'w') as f:
        json.dump(enriched_meta, f, indent=2)

    print(f"✅ 已保存 enriched metadata 至：{output_path}")